In [2]:
%pip install streamlit joblib pandas scikit-learn lightgbm

  Using cached streamlit-1.67.0-py3-none-any.whl.metadata (10 kB)
  Using cached altair-6.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached click-8.5.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached pydeck-0.9.3-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached protobuf-7.36.2-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached pyarrow-26.0.1-cp315-cp315-win_amd64.whl.metadata (3.0 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached starlette-1.10.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached anyio-4.16.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached python_multipart-0.0.34-py3-none-any.whl.metadata (6.3 kB)
  Using cached websockets-17.4-cp315-cp315-win_amd64.whl.metadata (6.5 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## streamlit 패키지 설치

In [3]:
%%writefile app.py
import streamlit as st
import joblib
import pandas as pd

st.title("당뇨 건강지표 기반 위험도 예측 대시보드")

Writing app.py


In [4]:
!streamlit run app.py

^C


## 파월셀에서 app.py 실행 구동

In [6]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split

# 1. 데이터 로드 및 파생 변수 생성
file_path = "../data/processed/diabetes_binary_health_indicators_cleaned.csv"
df = pd.read_csv(file_path)

target_col = "Diabetes_binary"
y = df[target_col].values
X_raw = df.drop(columns=[target_col])


def add_interaction_features(data: pd.DataFrame) -> pd.DataFrame:
  d = data.copy()

  is_obese = (d["BMI"] >= 30).astype(int) if "BMI" in d.columns else None
  age_60plus = (d["Age"] >= 9).astype(int) if "Age" in d.columns else None
  genhlth_risk = (
      (d["GenHlth"] >= 3).astype(int) if "GenHlth" in d.columns else None
  )

  metabolic_parts = []
  if "HighBP" in d.columns:
    metabolic_parts.append(d["HighBP"])
  if "HighChol" in d.columns:
    metabolic_parts.append(d["HighChol"])
  if is_obese is not None:
    metabolic_parts.append(is_obese)

  if metabolic_parts:
    d["Metabolic_Risk_Count"] = sum(metabolic_parts)
    if len(metabolic_parts) == 3:
      d["Triple_Metabolic_Risk"] = (d["Metabolic_Risk_Count"] == 3).astype(int)

  if {"Age", "HighBP"}.issubset(d.columns):
    d["Age_HighBP_Inter"] = d["Age"] * d["HighBP"]
  if age_60plus is not None and genhlth_risk is not None:
    d["Aged_Poor_Health"] = age_60plus * genhlth_risk

  cardio_col = (
      "CardioDisease"
      if "CardioDisease" in d.columns
      else ("HeartDiseaseorAttack" if "HeartDiseaseorAttack" in d.columns else None)
  )
  if cardio_col and "DiffWalk" in d.columns:
    d["Cardio_DiffWalk_Inter"] = d[cardio_col] * d["DiffWalk"]
  if {"GenHlth", "BMI"}.issubset(d.columns):
    d["GenHlth_BMI_Inter"] = d["GenHlth"] * d["BMI"]

  life_parts = []
  if "PhysActivity" in d.columns:
    life_parts.append(1 - d["PhysActivity"])
  if "Smoker" in d.columns:
    life_parts.append(d["Smoker"])
  if "Fruits" in d.columns:
    life_parts.append(1 - d["Fruits"])
  if "Veggies" in d.columns:
    life_parts.append(1 - d["Veggies"])

  if life_parts:
    d["Unhealthy_Lifestyle_Score"] = sum(life_parts)
    if "Metabolic_Risk_Count" in d.columns:
      d["Metabolic_BadLife_Inter"] = (
          (d["Metabolic_Risk_Count"] >= 2)
          & (d["Unhealthy_Lifestyle_Score"] >= 3)
      ).astype(int)

  return d


X = add_interaction_features(X_raw)

# 2. 학습셋(80%)으로 최종 LightGBM 모델 학습 (테스트셋 20%는 평가 전용으로 분리 유지)
X_train, _, y_train, _ = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

final_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
final_model.fit(X_train, y_train)

# 3. ../models/ 폴더에 모델 번들 저장
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)
save_path = models_dir / "lgbm_diabetes_model.joblib"

bundle = {
    "model": final_model,
    "feature_columns": list(X_train.columns),
    "raw_columns": list(X_raw.columns),
    "thresholds": {
        "screening": 0.42,  # 1차 스크리닝 (High Recall)
        "baseline": 0.50,  # 표준 기준선
        "targeting": 0.65,  # 정밀 타겟팅 (F1 균형)
    },
}

joblib.dump(bundle, save_path)
print("저장 완료:", save_path.resolve())

저장 완료: C:\Users\USER\Documents\ai-campus-project\models\lgbm_diabetes_model.joblib


## 터미널에서 대시보드 구현


@'
from pathlib import Path
import joblib
import pandas as pd
import streamlit as st

st.set_page_config(page_title="당뇨 건강지표 예측 대시보드", layout="wide")


def add_interaction_features(data: pd.DataFrame) -> pd.DataFrame:
    d = data.copy()

    is_obese = (d["BMI"] >= 30).astype(int) if "BMI" in d.columns else None
    age_60plus = (d["Age"] >= 9).astype(int) if "Age" in d.columns else None
    genhlth_risk = (d["GenHlth"] >= 3).astype(int) if "GenHlth" in d.columns else None

    metabolic_parts = []
    if "HighBP" in d.columns:
        metabolic_parts.append(d["HighBP"])
    if "HighChol" in d.columns:
        metabolic_parts.append(d["HighChol"])
    if is_obese is not None:
        metabolic_parts.append(is_obese)

    if metabolic_parts:
        d["Metabolic_Risk_Count"] = sum(metabolic_parts)
        if len(metabolic_parts) == 3:
            d["Triple_Metabolic_Risk"] = (d["Metabolic_Risk_Count"] == 3).astype(int)

    if {"Age", "HighBP"}.issubset(d.columns):
        d["Age_HighBP_Inter"] = d["Age"] * d["HighBP"]
    if age_60plus is not None and genhlth_risk is not None:
        d["Aged_Poor_Health"] = age_60plus * genhlth_risk

    cardio_col = (
        "CardioDisease"
        if "CardioDisease" in d.columns
        else ("HeartDiseaseorAttack" if "HeartDiseaseorAttack" in d.columns else None)
    )
    if cardio_col and "DiffWalk" in d.columns:
        d["Cardio_DiffWalk_Inter"] = d[cardio_col] * d["DiffWalk"]
    if {"GenHlth", "BMI"}.issubset(d.columns):
        d["GenHlth_BMI_Inter"] = d["GenHlth"] * d["BMI"]

    life_parts = []
    if "PhysActivity" in d.columns:
        life_parts.append(1 - d["PhysActivity"])
    if "Smoker" in d.columns:
        life_parts.append(d["Smoker"])
    if "Fruits" in d.columns:
        life_parts.append(1 - d["Fruits"])
    if "Veggies" in d.columns:
        life_parts.append(1 - d["Veggies"])

    if life_parts:
        d["Unhealthy_Lifestyle_Score"] = sum(life_parts)
        if "Metabolic_Risk_Count" in d.columns:
            d["Metabolic_BadLife_Inter"] = (
                (d["Metabolic_Risk_Count"] >= 2) & (d["Unhealthy_Lifestyle_Score"] >= 3)
            ).astype(int)

    return d


@st.cache_resource
def load_model_bundle(model_path: str):
    return joblib.load(model_path)


st.title("당뇨 건강지표 기반 위험도 예측 대시보드")
st.caption("학습된 LightGBM 모델과 확정 임계값(0.42 / 0.65)을 연동한 예측 화면입니다.")

model_file = Path(__file__).resolve().parent / "models" / "lgbm_diabetes_model.joblib"

st.sidebar.header("건강지표 입력")
gen_hlth = st.sidebar.slider("일반 건강 상태 (GenHlth, 1=매우좋음 ~ 5=매우나쁨)", 1, 5, 3)
high_bp = st.sidebar.selectbox("고혈압 여부 (HighBP)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
high_chol = st.sidebar.selectbox("고콜레스테롤 여부 (HighChol)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
bmi = st.sidebar.number_input("BMI", min_value=12.0, max_value=60.0, value=28.0, step=0.5)
age = st.sidebar.slider("연령 코드 (Age, 1~13)", 1, 13, 7)
diff_walk = st.sidebar.selectbox("보행 어려움 (DiffWalk)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
heart_disease = st.sidebar.selectbox("심혈관 질환 (HeartDiseaseorAttack)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
phys_activity = st.sidebar.selectbox("신체 활동 (PhysActivity)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
smoker = st.sidebar.selectbox("흡연 여부 (Smoker)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
fruits = st.sidebar.selectbox("과일 섭취 (Fruits)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
veggies = st.sidebar.selectbox("채소 섭취 (Veggies)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")

if not model_file.exists():
    st.warning(f"모델 파일이 아직 없습니다: `{model_file}`")
    st.stop()

bundle = load_model_bundle(str(model_file))
model = bundle["model"]
feature_columns = bundle["feature_columns"]
raw_columns = bundle.get("raw_columns", [])
thresholds = bundle.get("thresholds", {"screening": 0.42, "baseline": 0.50, "targeting": 0.65})

raw_input = {
    "HighBP": high_bp,
    "HighChol": high_chol,
    "BMI": bmi,
    "Smoker": smoker,
    "HeartDiseaseorAttack": heart_disease,
    "PhysActivity": phys_activity,
    "Fruits": fruits,
    "Veggies": veggies,
    "GenHlth": gen_hlth,
    "DiffWalk": diff_walk,
    "Age": age,
}

full_raw = {col: raw_input.get(col, 0) for col in raw_columns} if raw_columns else raw_input
raw_df = pd.DataFrame([full_raw])
fe_df = add_interaction_features(raw_df)
X_infer = fe_df.reindex(columns=feature_columns, fill_value=0)

prob = float(model.predict_proba(X_infer)[0][1])

th_screen = thresholds.get("screening", 0.42)
th_target = thresholds.get("targeting", 0.65)

col1, col2, col3 = st.columns(3)
col1.metric("예상 당뇨 위험 확률", f"{prob * 100:.1f}%")
col2.metric(
    f"1차 스크리닝 (≥ {th_screen:.2f})",
    "고위험군 선별" if prob >= th_screen else "일반 관찰",
)
col3.metric(
    f"정밀 타겟팅 (≥ {th_target:.2f})",
    "집중 개입 대상" if prob >= th_target else "비대상",
)

st.subheader("입력 지표 요약")
st.dataframe(pd.DataFrame([raw_input]), width="stretch")

if hasattr(model, "feature_importances_"):
    st.subheader("모델 주요 변수 중요도 (Top 10)")
    imp_df = (
        pd.DataFrame({
            "변수명": feature_columns,
            "중요도": model.feature_importances_,
        })
        .sort_values("중요도", ascending=False)
        .head(10)
    )
    st.dataframe(imp_df, width="stretch")
'@ | Out-File -FilePath .\app.py -Encoding utf8

.\.venv\Scripts\streamlit.exe run app.py

## 터미널에서 당뇨 위험도 진행바/게이지 시각화와 운영 기준선(0.42, 0.65) 설명 카드 추가. 
기존 예측·판정·변수 중요도 기능은 그대로 유지하면서 현재 확률이 어느 구간에 놓여 있는지 한눈에 보이도록 구성

@'
from pathlib import Path
import joblib
import pandas as pd
import streamlit as st

st.set_page_config(page_title="당뇨 건강지표 예측 대시보드", layout="wide")


def add_interaction_features(data: pd.DataFrame) -> pd.DataFrame:
    d = data.copy()

    is_obese = (d["BMI"] >= 30).astype(int) if "BMI" in d.columns else None
    age_60plus = (d["Age"] >= 9).astype(int) if "Age" in d.columns else None
    genhlth_risk = (d["GenHlth"] >= 3).astype(int) if "GenHlth" in d.columns else None

    metabolic_parts = []
    if "HighBP" in d.columns:
        metabolic_parts.append(d["HighBP"])
    if "HighChol" in d.columns:
        metabolic_parts.append(d["HighChol"])
    if is_obese is not None:
        metabolic_parts.append(is_obese)

    if metabolic_parts:
        d["Metabolic_Risk_Count"] = sum(metabolic_parts)
        if len(metabolic_parts) == 3:
            d["Triple_Metabolic_Risk"] = (d["Metabolic_Risk_Count"] == 3).astype(int)

    if {"Age", "HighBP"}.issubset(d.columns):
        d["Age_HighBP_Inter"] = d["Age"] * d["HighBP"]
    if age_60plus is not None and genhlth_risk is not None:
        d["Aged_Poor_Health"] = age_60plus * genhlth_risk

    cardio_col = (
        "CardioDisease"
        if "CardioDisease" in d.columns
        else ("HeartDiseaseorAttack" if "HeartDiseaseorAttack" in d.columns else None)
    )
    if cardio_col and "DiffWalk" in d.columns:
        d["Cardio_DiffWalk_Inter"] = d[cardio_col] * d["DiffWalk"]
    if {"GenHlth", "BMI"}.issubset(d.columns):
        d["GenHlth_BMI_Inter"] = d["GenHlth"] * d["BMI"]

    life_parts = []
    if "PhysActivity" in d.columns:
        life_parts.append(1 - d["PhysActivity"])
    if "Smoker" in d.columns:
        life_parts.append(d["Smoker"])
    if "Fruits" in d.columns:
        life_parts.append(1 - d["Fruits"])
    if "Veggies" in d.columns:
        life_parts.append(1 - d["Veggies"])

    if life_parts:
        d["Unhealthy_Lifestyle_Score"] = sum(life_parts)
        if "Metabolic_Risk_Count" in d.columns:
            d["Metabolic_BadLife_Inter"] = (
                (d["Metabolic_Risk_Count"] >= 2) & (d["Unhealthy_Lifestyle_Score"] >= 3)
            ).astype(int)

    return d


@st.cache_resource
def load_model_bundle(model_path: str):
    return joblib.load(model_path)


st.title("당뇨 건강지표 기반 위험도 예측 대시보드")
st.caption("학습된 LightGBM 모델과 확정 임계값(0.42 / 0.65)을 연동한 예측 화면입니다.")

model_file = Path(__file__).resolve().parent / "models" / "lgbm_diabetes_model.joblib"

st.sidebar.header("건강지표 입력")
gen_hlth = st.sidebar.slider("일반 건강 상태 (GenHlth, 1=매우좋음 ~ 5=매우나쁨)", 1, 5, 3)
high_bp = st.sidebar.selectbox("고혈압 여부 (HighBP)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
high_chol = st.sidebar.selectbox("고콜레스테롤 여부 (HighChol)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
bmi = st.sidebar.number_input("BMI", min_value=12.0, max_value=60.0, value=28.0, step=0.5)
age = st.sidebar.slider("연령 코드 (Age, 1~13)", 1, 13, 7)
diff_walk = st.sidebar.selectbox("보행 어려움 (DiffWalk)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
heart_disease = st.sidebar.selectbox("심혈관 질환 (HeartDiseaseorAttack)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
phys_activity = st.sidebar.selectbox("신체 활동 (PhysActivity)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
smoker = st.sidebar.selectbox("흡연 여부 (Smoker)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
fruits = st.sidebar.selectbox("과일 섭취 (Fruits)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
veggies = st.sidebar.selectbox("채소 섭취 (Veggies)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")

if not model_file.exists():
    st.warning(f"모델 파일이 아직 없습니다: `{model_file}`")
    st.stop()

bundle = load_model_bundle(str(model_file))
model = bundle["model"]
feature_columns = bundle["feature_columns"]
raw_columns = bundle.get("raw_columns", [])
thresholds = bundle.get("thresholds", {"screening": 0.42, "baseline": 0.50, "targeting": 0.65})

raw_input = {
    "HighBP": high_bp,
    "HighChol": high_chol,
    "BMI": bmi,
    "Smoker": smoker,
    "HeartDiseaseorAttack": heart_disease,
    "PhysActivity": phys_activity,
    "Fruits": fruits,
    "Veggies": veggies,
    "GenHlth": gen_hlth,
    "DiffWalk": diff_walk,
    "Age": age,
}

full_raw = {col: raw_input.get(col, 0) for col in raw_columns} if raw_columns else raw_input
raw_df = pd.DataFrame([full_raw])
fe_df = add_interaction_features(raw_df)
X_infer = fe_df.reindex(columns=feature_columns, fill_value=0)

prob = float(model.predict_proba(X_infer)[0][1])
prob_pct = prob * 100.0

th_screen = float(thresholds.get("screening", 0.42))
th_target = float(thresholds.get("targeting", 0.65))

# 상단 핵심 지표
col1, col2, col3 = st.columns(3)
col1.metric("예상 당뇨 위험 확률", f"{prob_pct:.1f}%")
col2.metric(
    f"1차 스크리닝 (≥ {th_screen:.2f})",
    "고위험군 선별" if prob >= th_screen else "일반 관찰",
)
col3.metric(
    f"정밀 타겟팅 (≥ {th_target:.2f})",
    "집중 개입 대상" if prob >= th_target else "비대상",
)

# 위험도 게이지(진행바 + 구간 표시)
st.subheader("위험도 게이지 및 운영 구간")
progress_val = min(max(prob, 0.0), 1.0)
st.progress(progress_val, text=f"현재 예측 위험도: {prob_pct:.1f}% (1차 스크리닝 기준 {th_screen*100:.0f}% / 정밀 타겟팅 기준 {th_target*100:.0f}%)")

if prob >= th_target:
    st.error(f"현재 구간: **정밀 타겟팅 구간 (≥ {th_target:.2f})** — 오탐을 줄이고 집중 개입이 필요한 고위험군입니다.")
elif prob >= th_screen:
    st.warning(f"현재 구간: **1차 스크리닝 구간 ({th_screen:.2f} ~ {th_target:.2f} 미만)** — 미탐을 줄이기 위해 폭넓게 선별하는 1차 관찰·상담 대상입니다.")
else:
    st.info(f"현재 구간: **일반 관찰 구간 (< {th_screen:.2f})** — 두 운영 임계값 기준선 아래로 일반 건강관리 흐름에 해당합니다.")

# 운영 기준선(0.42, 0.65) 설명 카드
st.subheader("운영 임계값 기준선 안내")
guide_col1, guide_col2 = st.columns(2)

with guide_col1:
    with st.container(border=True):
        st.markdown(f"### 1차 스크리닝 · 임계값 `{th_screen:.2f}`")
        st.markdown("- **목적:** 실제 당뇨 위험군을 놓치지 않도록 **미탐(FN) 최소화(High Recall)**")
        st.markdown("- **테스트셋 참고:** 약 **42.6%** 선별, **Recall 0.8580** 수준")
        st.markdown("- **활용:** 폭넓은 사전 안내, 생활습관 점검 및 1차 선별")

with guide_col2:
    with st.container(border=True):
        st.markdown(f"### 정밀 타겟팅 · 임계값 `{th_target:.2f}`")
        st.markdown("- **목적:** **오탐(FP) 및 개입 비용 절감**, 정밀 개입 대상 압축")
        st.markdown("- **테스트셋 참고:** 약 **22.6%** 선별, **Precision 0.3818 / F1 0.4724** 수준")
        st.markdown("- **활용:** 자원이 제한된 심화 상담·집중 관리 대상 선정")

st.subheader("입력 지표 요약")
st.dataframe(pd.DataFrame([raw_input]), width="stretch")

if hasattr(model, "feature_importances_"):
    st.subheader("모델 주요 변수 중요도 (Top 10)")
    imp_df = (
        pd.DataFrame({
            "변수명": feature_columns,
            "중요도": model.feature_importances_,
        })
        .sort_values("중요도", ascending=False)
        .head(10)
    )
    st.dataframe(imp_df, width="stretch")
'@ | Out-File -FilePath .\app.py -Encoding utf8

.\.venv\Scripts\streamlit.exe run app.py

## 터미널에서 실제 나이(18~120세)를 입력하면 BRFSS 기준 연령 코드(1~13)로 자동 변환되어 모델에 넘어가도록 app.py를 재구성

BRFSS 기준 연령 코드는 다음과 같이 1~13으로 구분:

1: 18~24세 | 2: 25~29세 | 3: 30~34세 | 4: 35~39세

5: 40~44세 | 6: 45~49세 | 7: 50~54세 | 8: 55~59세

9: 60~64세 | 10: 65~69세 | 11: 70~74세 | 12: 75~79세

13: 80세 이상
(18세 미만 입력 시 1로, 80세 이상 입력 시 13으로 안전하게 매핑되도록 처리.)



@'
from pathlib import Path
import joblib
import pandas as pd
import streamlit as st

st.set_page_config(page_title="당뇨 건강지표 예측 대시보드", layout="wide")


def age_to_brfss_code(real_age: int) -> int:
    """실제 나이(세)를 BRFSS 기준 연령 코드(1~13)로 변환"""
    if real_age < 25:
        return 1
    elif real_age < 30:
        return 2
    elif real_age < 35:
        return 3
    elif real_age < 40:
        return 4
    elif real_age < 45:
        return 5
    elif real_age < 50:
        return 6
    elif real_age < 55:
        return 7
    elif real_age < 60:
        return 8
    elif real_age < 65:
        return 9
    elif real_age < 70:
        return 10
    elif real_age < 75:
        return 11
    elif real_age < 80:
        return 12
    else:
        return 13


def add_interaction_features(data: pd.DataFrame) -> pd.DataFrame:
    d = data.copy()

    is_obese = (d["BMI"] >= 30).astype(int) if "BMI" in d.columns else None
    age_60plus = (d["Age"] >= 9).astype(int) if "Age" in d.columns else None
    genhlth_risk = (d["GenHlth"] >= 3).astype(int) if "GenHlth" in d.columns else None

    metabolic_parts = []
    if "HighBP" in d.columns:
        metabolic_parts.append(d["HighBP"])
    if "HighChol" in d.columns:
        metabolic_parts.append(d["HighChol"])
    if is_obese is not None:
        metabolic_parts.append(is_obese)

    if metabolic_parts:
        d["Metabolic_Risk_Count"] = sum(metabolic_parts)
        if len(metabolic_parts) == 3:
            d["Triple_Metabolic_Risk"] = (d["Metabolic_Risk_Count"] == 3).astype(int)

    if {"Age", "HighBP"}.issubset(d.columns):
        d["Age_HighBP_Inter"] = d["Age"] * d["HighBP"]
    if age_60plus is not None and genhlth_risk is not None:
        d["Aged_Poor_Health"] = age_60plus * genhlth_risk

    cardio_col = (
        "CardioDisease"
        if "CardioDisease" in d.columns
        else ("HeartDiseaseorAttack" if "HeartDiseaseorAttack" in d.columns else None)
    )
    if cardio_col and "DiffWalk" in d.columns:
        d["Cardio_DiffWalk_Inter"] = d[cardio_col] * d["DiffWalk"]
    if {"GenHlth", "BMI"}.issubset(d.columns):
        d["GenHlth_BMI_Inter"] = d["GenHlth"] * d["BMI"]

    life_parts = []
    if "PhysActivity" in d.columns:
        life_parts.append(1 - d["PhysActivity"])
    if "Smoker" in d.columns:
        life_parts.append(d["Smoker"])
    if "Fruits" in d.columns:
        life_parts.append(1 - d["Fruits"])
    if "Veggies" in d.columns:
        life_parts.append(1 - d["Veggies"])

    if life_parts:
        d["Unhealthy_Lifestyle_Score"] = sum(life_parts)
        if "Metabolic_Risk_Count" in d.columns:
            d["Metabolic_BadLife_Inter"] = (
                (d["Metabolic_Risk_Count"] >= 2) & (d["Unhealthy_Lifestyle_Score"] >= 3)
            ).astype(int)

    return d


@st.cache_resource
def load_model_bundle(model_path: str):
    return joblib.load(model_path)


st.title("당뇨 건강지표 기반 위험도 예측 대시보드")
st.caption("학습된 LightGBM 모델과 확정 임계값(0.42 / 0.65)을 연동한 예측 화면입니다.")

model_file = Path(__file__).resolve().parent / "models" / "lgbm_diabetes_model.joblib"

# 사이드바 건강지표 입력
st.sidebar.header("건강지표 입력")
real_age = st.sidebar.number_input("실제 나이 (세)", min_value=18, max_value=110, value=52, step=1)
age_code = age_to_brfss_code(int(real_age))
st.sidebar.caption(f"자동 변환된 연령 코드 (Age): **{age_code}** / 13")

gen_hlth = st.sidebar.slider("일반 건강 상태 (GenHlth, 1=매우좋음 ~ 5=매우나쁨)", 1, 5, 3)
high_bp = st.sidebar.selectbox("고혈압 여부 (HighBP)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
high_chol = st.sidebar.selectbox("고콜레스테롤 여부 (HighChol)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
bmi = st.sidebar.number_input("BMI", min_value=12.0, max_value=60.0, value=28.0, step=0.5)
diff_walk = st.sidebar.selectbox("보행 어려움 (DiffWalk)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
heart_disease = st.sidebar.selectbox("심혈관 질환 (HeartDiseaseorAttack)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
phys_activity = st.sidebar.selectbox("신체 활동 (PhysActivity)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
smoker = st.sidebar.selectbox("흡연 여부 (Smoker)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
fruits = st.sidebar.selectbox("과일 섭취 (Fruits)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
veggies = st.sidebar.selectbox("채소 섭취 (Veggies)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")

if not model_file.exists():
    st.warning(f"모델 파일이 아직 없습니다: `{model_file}`")
    st.stop()

bundle = load_model_bundle(str(model_file))
model = bundle["model"]
feature_columns = bundle["feature_columns"]
raw_columns = bundle.get("raw_columns", [])
thresholds = bundle.get("thresholds", {"screening": 0.42, "baseline": 0.50, "targeting": 0.65})

raw_input = {
    "HighBP": high_bp,
    "HighChol": high_chol,
    "BMI": bmi,
    "Smoker": smoker,
    "HeartDiseaseorAttack": heart_disease,
    "PhysActivity": phys_activity,
    "Fruits": fruits,
    "Veggies": veggies,
    "GenHlth": gen_hlth,
    "DiffWalk": diff_walk,
    "Age": age_code,
}

full_raw = {col: raw_input.get(col, 0) for col in raw_columns} if raw_columns else raw_input
raw_df = pd.DataFrame([full_raw])
fe_df = add_interaction_features(raw_df)
X_infer = fe_df.reindex(columns=feature_columns, fill_value=0)

prob = float(model.predict_proba(X_infer)[0][1])
prob_pct = prob * 100.0

th_screen = float(thresholds.get("screening", 0.42))
th_target = float(thresholds.get("targeting", 0.65))

# 상단 핵심 지표
col1, col2, col3 = st.columns(3)
col1.metric("예상 당뇨 위험 확률", f"{prob_pct:.1f}%")
col2.metric(
    f"1차 스크리닝 (≥ {th_screen:.2f})",
    "고위험군 선별" if prob >= th_screen else "일반 관찰",
)
col3.metric(
    f"정밀 타겟팅 (≥ {th_target:.2f})",
    "집중 개입 대상" if prob >= th_target else "비대상",
)

# 위험도 게이지(진행바 + 구간 표시)
st.subheader("위험도 게이지 및 운영 구간")
progress_val = min(max(prob, 0.0), 1.0)
st.progress(progress_val, text=f"현재 예측 위험도: {prob_pct:.1f}% (1차 스크리닝 기준 {th_screen*100:.0f}% / 정밀 타겟팅 기준 {th_target*100:.0f}%)")

if prob >= th_target:
    st.error(f"현재 구간: **정밀 타겟팅 구간 (≥ {th_target:.2f})** — 오탐을 줄이고 집중 개입이 필요한 고위험군입니다.")
elif prob >= th_screen:
    st.warning(f"현재 구간: **1차 스크리닝 구간 ({th_screen:.2f} ~ {th_target:.2f} 미만)** — 미탐을 줄이기 위해 폭넓게 선별하는 1차 관찰·상담 대상입니다.")
else:
    st.info(f"현재 구간: **일반 관찰 구간 (< {th_screen:.2f})** — 두 운영 임계값 기준선 아래로 일반 건강관리 흐름에 해당합니다.")

# 운영 기준선(0.42, 0.65) 설명 카드
st.subheader("운영 임계값 기준선 안내")
guide_col1, guide_col2 = st.columns(2)

with guide_col1:
    with st.container(border=True):
        st.markdown(f"### 1차 스크리닝 · 임계값 `{th_screen:.2f}`")
        st.markdown("- **목적:** 실제 당뇨 위험군을 놓치지 않도록 **미탐(FN) 최소화(High Recall)**")
        st.markdown("- **테스트셋 참고:** 약 **42.6%** 선별, **Recall 0.8580** 수준")
        st.markdown("- **활용:** 폭넓은 사전 안내, 생활습관 점검 및 1차 선별")

with guide_col2:
    with st.container(border=True):
        st.markdown(f"### 정밀 타겟팅 · 임계값 `{th_target:.2f}`")
        st.markdown("- **목적:** **오탐(FP) 및 개입 비용 절감**, 정밀 개입 대상 압축")
        st.markdown("- **테스트셋 참고:** 약 **22.6%** 선별, **Precision 0.3818 / F1 0.4724** 수준")
        st.markdown("- **활용:** 자원이 제한된 심화 상담·집중 관리 대상 선정")

st.subheader("입력 지표 요약 (실제 나이 및 변환 연령 코드 포함)")
summary_df = pd.DataFrame([{"Real_Age": int(real_age), **raw_input}])
st.dataframe(summary_df, width="stretch")

if hasattr(model, "feature_importances_"):
    st.subheader("모델 주요 변수 중요도 (Top 10)")
    imp_df = (
        pd.DataFrame({
            "변수명": feature_columns,
            "중요도": model.feature_importances_,
        })
        .sort_values("중요도", ascending=False)
        .head(10)
    )
    st.dataframe(imp_df, width="stretch")
'@ | Out-File -FilePath .\app.py -Encoding utf8

.\.venv\Scripts\streamlit.exe run app.py

## 터미널에서 사이드바의 나이 입력 라벨을 "만 나이 (세)"로 변경해 만 나이 기준 입력이라는 점을 명확히 표시하도록 app.py를 업데이트

@'
from pathlib import Path
import joblib
import pandas as pd
import streamlit as st

st.set_page_config(page_title="당뇨 건강지표 예측 대시보드", layout="wide")


def age_to_brfss_code(real_age: int) -> int:
    """만 나이(세)를 BRFSS 기준 연령 코드(1~13)로 변환"""
    if real_age < 25:
        return 1
    elif real_age < 30:
        return 2
    elif real_age < 35:
        return 3
    elif real_age < 40:
        return 4
    elif real_age < 45:
        return 5
    elif real_age < 50:
        return 6
    elif real_age < 55:
        return 7
    elif real_age < 60:
        return 8
    elif real_age < 65:
        return 9
    elif real_age < 70:
        return 10
    elif real_age < 75:
        return 11
    elif real_age < 80:
        return 12
    else:
        return 13


def add_interaction_features(data: pd.DataFrame) -> pd.DataFrame:
    d = data.copy()

    is_obese = (d["BMI"] >= 30).astype(int) if "BMI" in d.columns else None
    age_60plus = (d["Age"] >= 9).astype(int) if "Age" in d.columns else None
    genhlth_risk = (d["GenHlth"] >= 3).astype(int) if "GenHlth" in d.columns else None

    metabolic_parts = []
    if "HighBP" in d.columns:
        metabolic_parts.append(d["HighBP"])
    if "HighChol" in d.columns:
        metabolic_parts.append(d["HighChol"])
    if is_obese is not None:
        metabolic_parts.append(is_obese)

    if metabolic_parts:
        d["Metabolic_Risk_Count"] = sum(metabolic_parts)
        if len(metabolic_parts) == 3:
            d["Triple_Metabolic_Risk"] = (d["Metabolic_Risk_Count"] == 3).astype(int)

    if {"Age", "HighBP"}.issubset(d.columns):
        d["Age_HighBP_Inter"] = d["Age"] * d["HighBP"]
    if age_60plus is not None and genhlth_risk is not None:
        d["Aged_Poor_Health"] = age_60plus * genhlth_risk

    cardio_col = (
        "CardioDisease"
        if "CardioDisease" in d.columns
        else ("HeartDiseaseorAttack" if "HeartDiseaseorAttack" in d.columns else None)
    )
    if cardio_col and "DiffWalk" in d.columns:
        d["Cardio_DiffWalk_Inter"] = d[cardio_col] * d["DiffWalk"]
    if {"GenHlth", "BMI"}.issubset(d.columns):
        d["GenHlth_BMI_Inter"] = d["GenHlth"] * d["BMI"]

    life_parts = []
    if "PhysActivity" in d.columns:
        life_parts.append(1 - d["PhysActivity"])
    if "Smoker" in d.columns:
        life_parts.append(d["Smoker"])
    if "Fruits" in d.columns:
        life_parts.append(1 - d["Fruits"])
    if "Veggies" in d.columns:
        life_parts.append(1 - d["Veggies"])

    if life_parts:
        d["Unhealthy_Lifestyle_Score"] = sum(life_parts)
        if "Metabolic_Risk_Count" in d.columns:
            d["Metabolic_BadLife_Inter"] = (
                (d["Metabolic_Risk_Count"] >= 2) & (d["Unhealthy_Lifestyle_Score"] >= 3)
            ).astype(int)

    return d


@st.cache_resource
def load_model_bundle(model_path: str):
    return joblib.load(model_path)


st.title("당뇨 건강지표 기반 위험도 예측 대시보드")
st.caption("학습된 LightGBM 모델과 확정 임계값(0.42 / 0.65)을 연동한 예측 화면입니다.")

model_file = Path(__file__).resolve().parent / "models" / "lgbm_diabetes_model.joblib"

# 사이드바 건강지표 입력
st.sidebar.header("건강지표 입력")
real_age = st.sidebar.number_input("만 나이 (세)", min_value=18, max_value=110, value=52, step=1)
age_code = age_to_brfss_code(int(real_age))
st.sidebar.caption(f"자동 변환된 연령 코드 (Age): **{age_code}** / 13")

gen_hlth = st.sidebar.slider("일반 건강 상태 (GenHlth, 1=매우좋음 ~ 5=매우나쁨)", 1, 5, 3)
high_bp = st.sidebar.selectbox("고혈압 여부 (HighBP)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
high_chol = st.sidebar.selectbox("고콜레스테롤 여부 (HighChol)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
bmi = st.sidebar.number_input("BMI", min_value=12.0, max_value=60.0, value=28.0, step=0.5)
diff_walk = st.sidebar.selectbox("보행 어려움 (DiffWalk)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
heart_disease = st.sidebar.selectbox("심혈관 질환 (HeartDiseaseorAttack)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
phys_activity = st.sidebar.selectbox("신체 활동 (PhysActivity)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
smoker = st.sidebar.selectbox("흡연 여부 (Smoker)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
fruits = st.sidebar.selectbox("과일 섭취 (Fruits)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
veggies = st.sidebar.selectbox("채소 섭취 (Veggies)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")

if not model_file.exists():
    st.warning(f"모델 파일이 아직 없습니다: `{model_file}`")
    st.stop()

bundle = load_model_bundle(str(model_file))
model = bundle["model"]
feature_columns = bundle["feature_columns"]
raw_columns = bundle.get("raw_columns", [])
thresholds = bundle.get("thresholds", {"screening": 0.42, "baseline": 0.50, "targeting": 0.65})

raw_input = {
    "HighBP": high_bp,
    "HighChol": high_chol,
    "BMI": bmi,
    "Smoker": smoker,
    "HeartDiseaseorAttack": heart_disease,
    "PhysActivity": phys_activity,
    "Fruits": fruits,
    "Veggies": veggies,
    "GenHlth": gen_hlth,
    "DiffWalk": diff_walk,
    "Age": age_code,
}

full_raw = {col: raw_input.get(col, 0) for col in raw_columns} if raw_columns else raw_input
raw_df = pd.DataFrame([full_raw])
fe_df = add_interaction_features(raw_df)
X_infer = fe_df.reindex(columns=feature_columns, fill_value=0)

prob = float(model.predict_proba(X_infer)[0][1])
prob_pct = prob * 100.0

th_screen = float(thresholds.get("screening", 0.42))
th_target = float(thresholds.get("targeting", 0.65))

col1, col2, col3 = st.columns(3)
col1.metric("예상 당뇨 위험 확률", f"{prob_pct:.1f}%")
col2.metric(
    f"1차 스크리닝 (≥ {th_screen:.2f})",
    "고위험군 선별" if prob >= th_screen else "일반 관찰",
)
col3.metric(
    f"정밀 타겟팅 (≥ {th_target:.2f})",
    "집중 개입 대상" if prob >= th_target else "비대상",
)

st.subheader("위험도 게이지 및 운영 구간")
progress_val = min(max(prob, 0.0), 1.0)
st.progress(progress_val, text=f"현재 예측 위험도: {prob_pct:.1f}% (1차 스크리닝 기준 {th_screen*100:.0f}% / 정밀 타겟팅 기준 {th_target*100:.0f}%)")

if prob >= th_target:
    st.error(f"현재 구간: **정밀 타겟팅 구간 (≥ {th_target:.2f})** — 오탐을 줄이고 집중 개입이 필요한 고위험군입니다.")
elif prob >= th_screen:
    st.warning(f"현재 구간: **1차 스크리닝 구간 ({th_screen:.2f} ~ {th_target:.2f} 미만)** — 미탐을 줄이기 위해 폭넓게 선별하는 1차 관찰·상담 대상입니다.")
else:
    st.info(f"현재 구간: **일반 관찰 구간 (< {th_screen:.2f})** — 두 운영 임계값 기준선 아래로 일반 건강관리 흐름에 해당합니다.")

st.subheader("운영 임계값 기준선 안내")
guide_col1, guide_col2 = st.columns(2)

with guide_col1:
    with st.container(border=True):
        st.markdown(f"### 1차 스크리닝 · 임계값 `{th_screen:.2f}`")
        st.markdown("- **목적:** 실제 당뇨 위험군을 놓치지 않도록 **미탐(FN) 최소화(High Recall)**")
        st.markdown("- **테스트셋 참고:** 약 **42.6%** 선별, **Recall 0.8580** 수준")
        st.markdown("- **활용:** 폭넓은 사전 안내, 생활습관 점검 및 1차 선별")

with guide_col2:
    with st.container(border=True):
        st.markdown(f"### 정밀 타겟팅 · 임계값 `{th_target:.2f}`")
        st.markdown("- **목적:** **오탐(FP) 및 개입 비용 절감**, 정밀 개입 대상 압축")
        st.markdown("- **테스트셋 참고:** 약 **22.6%** 선별, **Precision 0.3818 / F1 0.4724** 수준")
        st.markdown("- **활용:** 자원이 제한된 심화 상담·집중 관리 대상 선정")

st.subheader("입력 지표 요약 (만 나이 및 변환 연령 코드 포함)")
summary_df = pd.DataFrame([{"Age_Years": int(real_age), **raw_input}])
st.dataframe(summary_df, width="stretch")

if hasattr(model, "feature_importances_"):
    st.subheader("모델 주요 변수 중요도 (Top 10)")
    imp_df = (
        pd.DataFrame({
            "변수명": feature_columns,
            "중요도": model.feature_importances_,
        })
        .sort_values("중요도", ascending=False)
        .head(10)
    )
    st.dataframe(imp_df, width="stretch")
'@ | Out-File -FilePath .\app.py -Encoding utf8

.\.venv\Scripts\streamlit.exe run app.py

## 터미널에서 "현재 입력값 기준 위험 요인 요약" 구성
핵심 지표 3칸: 감지된 위험 요인 개수, 대사 위험 점수(Metabolic_Risk_Count), 불건강 생활습관 점수(Unhealthy_Lifestyle_Score)를 한눈에 표시.

현재 감지된 위험 요인 리스트: 고혈압, 고콜레스테롤, 비만(BMI >= 30), 만 60세 이상(Age >= 9), 나쁜 일반 건강(GenHlth >= 3), 보행 어려움, 심혈관 질환, 운동 부족, 흡연, 과일/채소 섭취 부족 중 현재 조건에 해당하는 항목만 추려 보여줌.

생활·관리 체크 포인트: 현재 입력된 위험 요인에 맞춰 먼저 점검해 볼 수 있는 간단한 관리 포인트를 안내.

@'
from pathlib import Path
import joblib
import pandas as pd
import streamlit as st

st.set_page_config(page_title="당뇨 건강지표 예측 대시보드", layout="wide")


def age_to_brfss_code(real_age: int) -> int:
    """만 나이(세)를 BRFSS 기준 연령 코드(1~13)로 변환"""
    if real_age < 25:
        return 1
    elif real_age < 30:
        return 2
    elif real_age < 35:
        return 3
    elif real_age < 40:
        return 4
    elif real_age < 45:
        return 5
    elif real_age < 50:
        return 6
    elif real_age < 55:
        return 7
    elif real_age < 60:
        return 8
    elif real_age < 65:
        return 9
    elif real_age < 70:
        return 10
    elif real_age < 75:
        return 11
    elif real_age < 80:
        return 12
    else:
        return 13


def add_interaction_features(data: pd.DataFrame) -> pd.DataFrame:
    d = data.copy()

    is_obese = (d["BMI"] >= 30).astype(int) if "BMI" in d.columns else None
    age_60plus = (d["Age"] >= 9).astype(int) if "Age" in d.columns else None
    genhlth_risk = (d["GenHlth"] >= 3).astype(int) if "GenHlth" in d.columns else None

    metabolic_parts = []
    if "HighBP" in d.columns:
        metabolic_parts.append(d["HighBP"])
    if "HighChol" in d.columns:
        metabolic_parts.append(d["HighChol"])
    if is_obese is not None:
        metabolic_parts.append(is_obese)

    if metabolic_parts:
        d["Metabolic_Risk_Count"] = sum(metabolic_parts)
        if len(metabolic_parts) == 3:
            d["Triple_Metabolic_Risk"] = (d["Metabolic_Risk_Count"] == 3).astype(int)

    if {"Age", "HighBP"}.issubset(d.columns):
        d["Age_HighBP_Inter"] = d["Age"] * d["HighBP"]
    if age_60plus is not None and genhlth_risk is not None:
        d["Aged_Poor_Health"] = age_60plus * genhlth_risk

    cardio_col = (
        "CardioDisease"
        if "CardioDisease" in d.columns
        else ("HeartDiseaseorAttack" if "HeartDiseaseorAttack" in d.columns else None)
    )
    if cardio_col and "DiffWalk" in d.columns:
        d["Cardio_DiffWalk_Inter"] = d[cardio_col] * d["DiffWalk"]
    if {"GenHlth", "BMI"}.issubset(d.columns):
        d["GenHlth_BMI_Inter"] = d["GenHlth"] * d["BMI"]

    life_parts = []
    if "PhysActivity" in d.columns:
        life_parts.append(1 - d["PhysActivity"])
    if "Smoker" in d.columns:
        life_parts.append(d["Smoker"])
    if "Fruits" in d.columns:
        life_parts.append(1 - d["Fruits"])
    if "Veggies" in d.columns:
        life_parts.append(1 - d["Veggies"])

    if life_parts:
        d["Unhealthy_Lifestyle_Score"] = sum(life_parts)
        if "Metabolic_Risk_Count" in d.columns:
            d["Metabolic_BadLife_Inter"] = (
                (d["Metabolic_Risk_Count"] >= 2) & (d["Unhealthy_Lifestyle_Score"] >= 3)
            ).astype(int)

    return d


def summarize_risk_factors(raw_input: dict, real_age: int) -> tuple[list[str], list[str], int, int]:
    """현재 입력값 기준 위험 요인 목록과 관리 포인트 반환"""
    factors = []
    tips = []

    if raw_input["HighBP"] == 1:
        factors.append("고혈압 진단 이력 (HighBP = 1)")
        tips.append("혈압 수치 주기적 측정 및 나트륨 섭취 관리")
    if raw_input["HighChol"] == 1:
        factors.append("고콜레스테롤 진단 이력 (HighChol = 1)")
        tips.append("포화지방·트랜스지방 섭취 줄이기 및 지질 검사 확인")
    if raw_input["BMI"] >= 30:
        factors.append(f"비만 범위 BMI ({raw_input['BMI']:.1f} ≥ 30.0)")
        tips.append("체중 감량 및 식단·유산소 운동 병행")
    elif raw_input["BMI"] >= 25:
        factors.append(f"과체중 범위 BMI ({raw_input['BMI']:.1f} ≥ 25.0)")
    if raw_input["GenHlth"] >= 4:
        factors.append(f"주관적 건강 상태 저하 (GenHlth = {raw_input['GenHlth']}/5, 나쁨~매우나쁨)")
        tips.append("전반적 컨디션 저하 원인 확인 및 정기 건강검진 권장")
    elif raw_input["GenHlth"] == 3:
        factors.append("주관적 건강 상태 보통 이하 (GenHlth = 3/5)")
    if real_age >= 60:
        factors.append(f"만 60세 이상 연령대 (만 {real_age}세, Age ≥ 9)")
        tips.append("연령대 고려 정기 혈당·대사 지표 모니터링")
    if raw_input["HeartDiseaseorAttack"] == 1:
        factors.append("심혈관 질환 이력 (HeartDiseaseorAttack = 1)")
        tips.append("심혈관·대사 동반 위험 관리 및 전문의 상담")
    if raw_input["DiffWalk"] == 1:
        factors.append("보행 어려움 (DiffWalk = 1)")
        tips.append("관절·보행 부담을 줄인 안전한 신체 활동 선택")
    if raw_input["PhysActivity"] == 0:
        factors.append("신체 활동 부족 (PhysActivity = 0)")
        tips.append("주 150분 내외 가벼운 걷기 등 규칙적 활동 습관화")
    if raw_input["Smoker"] == 1:
        factors.append("흡연 중 (Smoker = 1)")
        tips.append("금연 실천 및 금연 지원 프로그램 활용")
    if raw_input["Fruits"] == 0:
        factors.append("과일 섭취 부족 (Fruits = 0)")
    if raw_input["Veggies"] == 0:
        factors.append("채소 섭취 부족 (Veggies = 0)")

    if (raw_input["Fruits"] == 0 or raw_input["Veggies"] == 0):
        tips.append("식이섬유가 풍부한 채소·과일 섭취 늘리기")

    metabolic_score = (
        int(raw_input["HighBP"] == 1)
        + int(raw_input["HighChol"] == 1)
        + int(raw_input["BMI"] >= 30)
    )
    lifestyle_score = (
        int(raw_input["PhysActivity"] == 0)
        + int(raw_input["Smoker"] == 1)
        + int(raw_input["Fruits"] == 0)
        + int(raw_input["Veggies"] == 0)
    )

    # 중복 제거 순서 유지
    unique_tips = list(dict.fromkeys(tips))
    return factors, unique_tips, metabolic_score, lifestyle_score


@st.cache_resource
def load_model_bundle(model_path: str):
    return joblib.load(model_path)


st.title("당뇨 건강지표 기반 위험도 예측 대시보드")
st.caption("학습된 LightGBM 모델과 확정 임계값(0.42 / 0.65)을 연동한 예측 화면입니다.")

model_file = Path(__file__).resolve().parent / "models" / "lgbm_diabetes_model.joblib"

# 사이드바 건강지표 입력
st.sidebar.header("건강지표 입력")
real_age = st.sidebar.number_input("만 나이 (세)", min_value=18, max_value=110, value=52, step=1)
age_code = age_to_brfss_code(int(real_age))
st.sidebar.caption(f"자동 변환된 연령 코드 (Age): **{age_code}** / 13")

gen_hlth = st.sidebar.slider("일반 건강 상태 (GenHlth, 1=매우좋음 ~ 5=매우나쁨)", 1, 5, 3)
high_bp = st.sidebar.selectbox("고혈압 여부 (HighBP)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
high_chol = st.sidebar.selectbox("고콜레스테롤 여부 (HighChol)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
bmi = st.sidebar.number_input("BMI", min_value=12.0, max_value=60.0, value=28.0, step=0.5)
diff_walk = st.sidebar.selectbox("보행 어려움 (DiffWalk)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
heart_disease = st.sidebar.selectbox("심혈관 질환 (HeartDiseaseorAttack)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
phys_activity = st.sidebar.selectbox("신체 활동 (PhysActivity)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
smoker = st.sidebar.selectbox("흡연 여부 (Smoker)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
fruits = st.sidebar.selectbox("과일 섭취 (Fruits)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")
veggies = st.sidebar.selectbox("채소 섭취 (Veggies)", options=[0, 1], format_func=lambda x: "예 (1)" if x == 1 else "아니오 (0)")

if not model_file.exists():
    st.warning(f"모델 파일이 아직 없습니다: `{model_file}`")
    st.stop()

bundle = load_model_bundle(str(model_file))
model = bundle["model"]
feature_columns = bundle["feature_columns"]
raw_columns = bundle.get("raw_columns", [])
thresholds = bundle.get("thresholds", {"screening": 0.42, "baseline": 0.50, "targeting": 0.65})

raw_input = {
    "HighBP": high_bp,
    "HighChol": high_chol,
    "BMI": bmi,
    "Smoker": smoker,
    "HeartDiseaseorAttack": heart_disease,
    "PhysActivity": phys_activity,
    "Fruits": fruits,
    "Veggies": veggies,
    "GenHlth": gen_hlth,
    "DiffWalk": diff_walk,
    "Age": age_code,
}

full_raw = {col: raw_input.get(col, 0) for col in raw_columns} if raw_columns else raw_input
raw_df = pd.DataFrame([full_raw])
fe_df = add_interaction_features(raw_df)
X_infer = fe_df.reindex(columns=feature_columns, fill_value=0)

prob = float(model.predict_proba(X_infer)[0][1])
prob_pct = prob * 100.0

th_screen = float(thresholds.get("screening", 0.42))
th_target = float(thresholds.get("targeting",

## streamlit 대시보드상 구동 오류 수정

1단계: uv.lock에 lightgbm 추가
터미널(PowerShell)에서 uv로 lightgbm을 추가해 uv.lock과 pyproject.toml을 갱신. 
PowerShell
uv add lightgbm


2단계: requirements.txt도 같이 조정

Plaintext
streamlit
pandas
joblib
lightgbm
scikit-learn==1.7.2

In [8]:
GENHLTH_OPTIONS = {
    1: ("1점 — 최상 (Excellent)", "평소 컨디션이 매우 좋고 건강 상태가 최상인 경우"),
    2: ("2점 — 매우 좋음 (Very Good)", "전반적으로 건강 상태가 매우 양호한 경우"),
    3: ("3점 — 좋음 (Good)", "무난하게 좋은 편이나, 이 점수대부터 위험 구간(≥ 3) 진입"),
    4: ("4점 — 보통 (Fair)", "건강이 보통 수준이며 관리가 필요한 구간"),
    5: ("5점 — 나쁨 (Poor)", "건강 상태가 나쁘다고 느끼며 주의가 필요한 구간"),
}

gen_hlth = st.sidebar.selectbox(
    "일반 건강 상태 (GenHlth)",
    options=[1, 2, 3, 4, 5],
    index=2,  # 기본값 3점
    format_func=lambda x: GENHLTH_OPTIONS[x][0],
    help="1점(최상)부터 5점(나쁨)까지 선택할 수 있습니다.",
)

_, gen_hlth_desc = GENHLTH_OPTIONS[gen_hlth]
st.sidebar.caption(f"**선택 기준:** {gen_hlth_desc}")

with st.sidebar.expander("1~5점 기준 설명 보기"):
    for score in range(1, 6):
        label, desc = GENHLTH_OPTIONS[score]
        st.markdown(f"- **{label}:** {desc}")

2026-11-25 11:03:14.321 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-25 11:03:14.322 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-25 11:03:14.325 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-25 11:03:14.327 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-25 11:03:14.329 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-25 11:03:14.330 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-25 11:03:14.331 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-25 11:03:14.332 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar